In [11]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import joblib
import pandas as pd
import seaborn as sns
import optuna
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix
np.random.seed(42)

In [12]:
# Läs in träningsfilen
df = pd.read_csv(r"C:\Users\Eier\OneDrive\Dokument\DataScience\Data_science_projektkurs\archive (1)\diabetes_prediction_dataset.csv")
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df['gender'].unique())
print(df['smoking_history'].unique())
df_encoded = pd.get_dummies(df, columns=['gender', 'smoking_history'], drop_first=False)
print(df_encoded.head())
print(df_encoded.columns.tolist())
numeric_cols = ['age', 'hypertension', 'heart_disease', 'bmi', 
                'HbA1c_level', 'blood_glucose_level', 'diabetes']

df_numeric = df_encoded[numeric_cols]
print(df_numeric.describe().T)
print(df['diabetes'].value_counts())

Y = df_encoded['diabetes']
X = df_encoded.drop(columns=['diabetes'])

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  str    
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  str    
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 6.9 MB
None
                 age  hypertension  heart_disease            bmi  \
count  100000.000000  100000.00000  100000.000000  100000.000000   
mean       41.885856       0.07485       0.039420      27.320767   
std        22.516840       0.26315       0.194593       6.636783   
min   

In [13]:
from sklearn.model_selection import train_test_split

# 70% train, 30% temp
X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, Y, test_size=0.30, random_state=42, stratify=Y
)

# 15% val, 15% test
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.50, random_state=42, stratify=Y_temp
)

print(X_train.shape, Y_train.shape)
print(X_val.shape, Y_val.shape)
print(X_test.shape, Y_test.shape)

(70000, 15) (70000,)
(15000, 15) (15000,)
(15000, 15) (15000,)


Initering och träning av modell logistik regression, resultat sparas och kan återhämtas.

In [14]:
# Skapa pipeline med StandardScaler och baseline setting.

rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced_subsample',
        n_jobs=-1,
        random_state=42
    ))
])

rf.fit(X_train, Y_train)
score = rf.score(X_val, Y_val)

print("Modellen är tränad")
print("RandomForestClassifier:", score)

joblib.dump(rf, 'rf_trained.pkl')


Modellen är tränad
RandomForestClassifier: 0.9704666666666667


['rf_trained.pkl']

In [15]:
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score
import pandas as pd

def evaluate_thresholds(model, X_val, Y_val, model_name):
    probs = model.predict_proba(X_val)[:, 1]
    thresholds = [0.50, 0.40, 0.30, 0.25, 0.20, 0.15, 0.10]
    results = []
    for t in thresholds:
        pred = (probs >= t).astype(int)
        results.append({
        "Model": model_name,
        "Threshold": t,
        "Precision": precision_score(Y_val, pred),
        "Recall": recall_score(Y_val, pred),
        "F1": f1_score(Y_val, pred),
        "F2": fbeta_score(Y_val, pred, beta=2)
        })
    return pd.DataFrame(results)

rf_results = evaluate_thresholds(
rf,
X_val,
Y_val,
"random forest"
)
print(rf_results)

           Model  Threshold  Precision    Recall        F1        F2
0  random forest       0.50   0.934238  0.701961  0.801612  0.738693
1  random forest       0.40   0.875119  0.720000  0.790017  0.746463
2  random forest       0.30   0.781275  0.759216  0.770088  0.763527
3  random forest       0.25   0.718097  0.781176  0.748310  0.767689
4  random forest       0.20   0.647282  0.803137  0.716836  0.766238
5  random forest       0.15   0.582340  0.843137  0.688882  0.773827
6  random forest       0.10   0.495795  0.878431  0.633843  0.760973


In [6]:
# Ladda modellen
rf = joblib.load('rf_trained.pkl')

# Prediktioner
y_pred = rf.predict(X_test)

# Rapport
print("Classification Report för RandomForestClassifier")
print(classification_report(Y_test, y_pred))

Classification Report för RandomForestClassifier
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     13725
           1       0.94      0.68      0.79      1275

    accuracy                           0.97     15000
   macro avg       0.95      0.84      0.88     15000
weighted avg       0.97      0.97      0.97     15000



In [ ]:
Optimering av modellen RandomForest, med hjälp av Optuna. Vi optimerar hyperparametrarna: n_estimators, max_depth, min_samples_split, min_samples_leaf och max_features.

In [9]:

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

# -------------------------------------------------------
# Convergence stopper
# -------------------------------------------------------
class ConvergenceStopper:
    def __init__(self, threshold=0.002, patience=5):
        self.threshold = threshold
        self.patience = patience
        self.last_best = None
        self.bad_count = 0

    def __call__(self, study, trial):
        current = trial.value
        best = study.best_value

        if self.last_best is None:
            self.last_best = best
            return

        improvement = best - self.last_best

        if improvement < self.threshold:
            self.bad_count += 1
        else:
            self.bad_count = 0
            self.last_best = best

        if self.bad_count >= self.patience:
            print(f"\n[OPTUNA] {self.patience} dåliga trials i rad → stoppar studien.")
            study.stop()

# -------------------------------------------------------
# RF hyperparameter space
# -------------------------------------------------------
def rf_param_space(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "max_depth": trial.suggest_int("max_depth", 8, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "class_weight": trial.suggest_categorical(
        "class_weight",
        ["balanced", "balanced_subsample", None]
    ),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
    }

# -------------------------------------------------------
# Objective-funktion
# -------------------------------------------------------
def objective_rf(trial):
    params = rf_param_space(trial)

    model = RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        bootstrap=params["bootstrap"],
        random_state=535,
        n_jobs=-1
    )

    model.fit(X_train, Y_train)
    preds = model.predict(X_val)

    score = recall_score(Y_val, preds)

    trial.set_user_attr("best_model", model)

    return score

# -------------------------------------------------------
# Train/val split
# -------------------------------------------------------
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train, Y_train, test_size=0.2, random_state=535
)

# -------------------------------------------------------
# Kör Optuna-studien (Steg 1)
# -------------------------------------------------------
stopper = ConvergenceStopper(threshold=0.002, patience=5)

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=100, callbacks=[stopper])

# -------------------------------------------------------
# Spara bästa modellen
# -------------------------------------------------------
best_rf = study_rf.best_trial.user_attrs["best_model"]
joblib.dump(best_rf, "best_rf_model.joblib")

print("\n[OPTUNA] Bästa RF-parametrar:")
print(study_rf.best_params)
print("[JOBLIB] Sparad som best_rf_model.joblib")


[I 2026-10-08 15:44:04,483] A new study created in memory with name: no-name-a32f3897-a3b5-4551-8711-d91d0ab3e1ea
[I 2026-10-08 15:44:05,640] Trial 0 finished with value: 0.671875 and parameters: {'n_estimators': 244, 'max_depth': 12, 'min_samples_split': 11, 'min_samples_leaf': 9, 'max_features': 'log2', 'class_weight': None, 'bootstrap': True}. Best is trial 0 with value: 0.671875.
[I 2026-10-08 15:44:07,323] Trial 1 finished with value: 0.6791666666666667 and parameters: {'n_estimators': 226, 'max_depth': 13, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': None, 'class_weight': None, 'bootstrap': True}. Best is trial 1 with value: 0.6791666666666667.
[I 2026-10-08 15:44:09,113] Trial 2 finished with value: 0.6739583333333333 and parameters: {'n_estimators': 472, 'max_depth': 17, 'min_samples_split': 11, 'min_samples_leaf': 5, 'max_features': 'log2', 'class_weight': 'balanced_subsample', 'bootstrap': True}. Best is trial 1 with value: 0.6791666666666667.
[I 2026-10-08 


[OPTUNA] 5 dåliga trials i rad → stoppar studien.

[OPTUNA] Bästa RF-parametrar:
{'n_estimators': 226, 'max_depth': 13, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': None, 'class_weight': None, 'bootstrap': True}
[JOBLIB] Sparad som best_rf_model.joblib


In [8]:

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# -------------------------------------------------------
# Ladda modellen från joblib
# -------------------------------------------------------
rf_model = joblib.load("best_rf_model.joblib")
print("[JOBLIB] RF-modellen laddad.")

# -------------------------------------------------------
# Prediktioner
# -------------------------------------------------------
preds = rf_model.predict(X_test)
probs = rf_model.predict_proba(X_test)[:, 1]

# -------------------------------------------------------
# Metrics
# -------------------------------------------------------
acc = accuracy_score(Y_test, preds)
f1 = f1_score(Y_test, preds)
prec = precision_score(Y_test, preds)
rec = recall_score(Y_test, preds)
auc = roc_auc_score(Y_test, probs)
cm = confusion_matrix(Y_test, preds)

# -------------------------------------------------------
# Utskrifter
# -------------------------------------------------------
print("\n====================")
print("   RF– METRICS")
print("====================")

print(f"Accuracy:      {acc:.4f}")
print(f"F1-score:      {f1:.4f}")
print(f"Precision:     {prec:.4f}")
print(f"Recall:        {rec:.4f}")
print(f"ROC-AUC:       {auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(Y_test, preds))

[JOBLIB] RF-modellen laddad.

   RF– METRICS
Accuracy:      0.9703
F1-score:      0.7960
Precision:     0.9581
Recall:        0.6808
ROC-AUC:       0.9736

Confusion Matrix:
[[13687    38]
 [  407   868]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     13725
           1       0.96      0.68      0.80      1275

    accuracy                           0.97     15000
   macro avg       0.96      0.84      0.89     15000
weighted avg       0.97      0.97      0.97     15000



In [10]:
# Ladda den sparade modellen
rf_model = joblib.load('best_rf_model.joblib') 

# Definiera scorer
scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro'),
    'recall': make_scorer(recall_score, average='macro'),
    'f1': make_scorer(f1_score, average='macro')
}

# Beräkna cross-validation scores
cv_results = cross_validate(rf_model, X_train, Y_train, cv=5, scoring=scoring)

# Skapa en lista med resultaten för varje fold
rf_results = []
for fold in range(5):
    rf_results.append({
        'Fold': str(fold + 1),  # Ändrat till sträng och fold 1-5 för snyggare tabell
        'Accuracy': cv_results['test_accuracy'][fold],
        'Precision': cv_results['test_precision'][fold],
        'Recall': cv_results['test_recall'][fold],
        'F1-score': cv_results['test_f1'][fold]
    })

    # Beräkna Mean (Medelvärde) och lägg till i listan
rf_results.append({
    'Fold': 'Mean',
    'Accuracy': np.mean(cv_results['test_accuracy']),
    'Precision': np.mean(cv_results['test_precision']),
    'Recall': np.mean(cv_results['test_recall']),
    'F1-score': np.mean(cv_results['test_f1'])
})

# Beräkna Std (Standardavvikelse) och lägg till i listan
rf_results.append({
    'Fold': 'Std',
    'Accuracy': np.std(cv_results['test_accuracy']),
    'Precision': np.std(cv_results['test_precision']),
    'Recall': np.std(cv_results['test_recall']),
    'F1-score': np.std(cv_results['test_f1'])
})

# Konvertera listan till en DataFrame
rf_results_df = pd.DataFrame(rf_results)
print(rf_results_df)

# Spara DataFrame med joblib 
joblib.dump(rf_results_df, 'rf_cv_results.pkl')




   Fold  Accuracy  Precision    Recall  F1-score
0     1  0.971964   0.976717  0.842286  0.896239
1     2  0.973482   0.985200  0.845945  0.901579
2     3  0.971250   0.976925  0.837652  0.892991
3     4  0.971786   0.982143  0.837305  0.894385
4     5  0.969107   0.979130  0.822624  0.882609
5  Mean  0.971518   0.980023  0.837162  0.893561
6   Std  0.001415   0.003244  0.007937  0.006203


['rf_cv_results.pkl']